In [1]:
import pandas as pd

Partie A : Questions de cours
A1. Pourquoi lire un fichier avec dtype=str est-il l'équivalent du principe de la couche RAW ?
parce que le format string prend en charge tout type de donnée par conséquent il n'y aura pas de plantage du code au chargement ni de modification des données, la structure des données est intacte
A2. Que fait errors="coerce" ? À quelle fonction BigQuery correspond-il ? Quelle étape ne faut-il jamais oublier juste après ?
errors="coerce" rend null toute conversion impossible, il fait référence à la fonction SAFE_PARSE_DATE de BIGQUERY, signaler les valeurs devenues NaN après la conversion.
A3. Cite deux différences de comportement entre groupby en pandas et GROUP BY en SQL.
Le groupby en pandas exclus les NaN par défaut et c'est pas le cas en SQL
et la somme d'un groupe sans valeur renvoie 0 et non NULL comme en SQL

In [2]:
# Partie B : Du RAW au STAGING (produits_catalogue.csv)
# B1. 🧮 Lis le fichier proprement. Donne shape, le nombre de NaN par colonne et la répartition des valeurs de categorie, NaN compris. 
produits_catalogue = pd.read_csv('produits_catalogue.csv', sep=';', dtype='str')
print('le shape :',produits_catalogue.shape) 
print('\nle nombre de NaN par colonne :', produits_catalogue.isna().sum())
print('\nla répartition des valeurs de categorie :', produits_catalogue['categorie'].value_counts(dropna=False))
# Quels problèmes vois-tu ?
# Plusieurs :
# une valeur manquante dans la colonne prix, dans la répartition des catégorie audio et accessoire ne sont pas normalisé,
# plusieurs écritures différentes

# B2. 🧮 Nettoie les données en gardant les colonnes brutes intactes. Crée de nouvelles colonnes :
df = produits_catalogue.copy()

# categorie correction
df['categorie_new'] = df['categorie'].str.strip().str.lower() 

# prix_num, en gérant la virgule
df['prix_num'] = df['prix'].str.replace(",", ".", regex=False)
df['prix_num'] = pd.to_numeric(df['prix_num'], errors='coerce')

# # date_maj_ok, en gérant les deux formats.
d1 = pd.to_datetime(df["date_maj"], format="%Y-%m-%d", errors="coerce")
d2 = pd.to_datetime(df["date_maj"], format="%d/%m/%Y", errors="coerce")
df["date_maj_ok"] = d1.fillna(d2)          # prend d1, sinon d2

# Liste les product_id dont le prix ou la date est devenu invalide, et explique pourquoi pour chacun.
print('\nListe les product_id dont le prix ou la date est devenu invalide : ')
print(df.loc[ ((df['prix'].notna()) & (df['prix_num'].isna()))  | ((df['date_maj'].notna()) & (df['date_maj_ok'].isna())), ['product_id'] ])
# c'est P5 et P6 , P5 c'est écris gratuit dans prix alors qu'on s'attend à un prix numérique et P6 30 février n'existe pas comme date

# B3. 🧮 Les doublons :
# a) combien de lignes sont entièrement identiques ? Combien de doublons y a-t-il sur product_id ?
nb_ligne_identique = df.duplicated().sum()
nb_doublon_product_id = df.duplicated(subset=["product_id"]).sum()
print('\nnb_ligne_identique : ', nb_ligne_identique)
print('nb_doublon_product_id : ', nb_doublon_product_id)

# b) P2 apparaît deux fois avec deux prix différents. Laquelle de ces deux lignes faut-il garder, et pourquoi ? 
# Il faut garder la valeur la plus recente selon la date_maj, car cette colonne indique une mise à jour du prix 
avant = len(df)
df = df.sort_values("date_maj_ok", ascending=False)
df = df.drop_duplicates(subset=["product_id"], keep="first")
print("doublons supprimés :", avant - len(df))  

# c) Combien de produits restent ? Quel est le prix final de P2 ?
print('\nproduits restant : ',len(df)) # 6 produit

# B4. Exporte staging_produits.csv avec seulement ces colonnes, sans l'index : product_id, nom, categorie, prix_num, date_maj_ok.
produits_catalogue_nettoye = df[['product_id', 'nom', 'categorie_new', 'prix_num', 'date_maj_ok']]
produits_catalogue_nettoye.columns = ['product_id', 'nom', 'categorie', 'prix_num', 'date_maj_ok']
produits_catalogue_nettoye.to_csv('staging_produits.csv', sep=';' , index=False)

le shape : (8, 5)

le nombre de NaN par colonne : product_id    0
nom           0
categorie     0
prix          1
date_maj      0
dtype: int64

la répartition des valeurs de categorie : categorie
audio          3
accessoire     2
 Accessoire    1
AUDIO          1
video          1
Name: count, dtype: int64

Liste les product_id dont le prix ou la date est devenu invalide : 
  product_id
4         P5
6         P6

nb_ligne_identique :  1
nb_doublon_product_id :  2
doublons supprimés : 2

produits restant :  6


In [35]:
# Partie C : Analyse (lignes_commande.csv + ton staging produits)
# Le montant d'une ligne de commande = quantite × prix_num. Pense à convertir quantite.
lignes_commande = pd.read_csv('lignes_commande.csv', sep=';' , dtype='str')
lignes_commande['quantite'] = pd.to_numeric(lignes_commande['quantite'], errors='coerce')
staging_produits = produits_catalogue_nettoye.copy()

# C1. 🧮 Fais un merge LEFT des lignes de commande avec les produits nettoyés, avec indicator=True. 
# Compare le nombre de lignes avant et après le merge : y a-t-il une duplication ? 
nb_ligne_avant = len(lignes_commande)
print('nb_ligne_avant : ',nb_ligne_avant) # 7
m1 = lignes_commande.merge(staging_produits, on='product_id', how='left', indicator=True)
print('nb_ligne_après : ',len(m1)) # 7 pas eu de dicplication
# Quelle ligne de commande est orpheline ?
orphelines = m1[m1['_merge'] == 'left_only'] # c'est la ligne O4 P9
print(orphelines)
# C2. 🧮 Calcule le chiffre d'affaires par catégorie, en ne gardant que les lignes dont le produit existe.
m2 = lignes_commande.merge(staging_produits, on='product_id', how='inner', indicator=True) # Je fais inner join sur product_id
m2['montant'] =   m2['quantite'] * m2['prix_num'] 
# Affiche aussi, pour chaque catégorie, le nombre de lignes et le nombre de montants connus.
# Trie par chiffre d'affaires décroissant. 
res = (m2.groupby('categorie')    
    .agg(ca=('montant', 'sum'),
         nb_ligne=('order_id', 'size'),
        nb_montant_connu=('montant', 'count'))
    .reset_index()
    .sort_values('ca', ascending=False))
print('\nAgrégation par catégorie  : \n', res)
# Quelle catégorie a un chiffre d'affaires incomplet, à cause de quelle ligne ?
# C'est la catégorie audio à cause de la ligne O6 de P4 car le prix_num est NaN
# C3. 🧮 Quels produits du catalogue n'ont jamais été commandés ?
m3 = staging_produits.merge(lignes_commande, on='product_id', how='left', indicator=True)
produit_jamais_vendu = m3[m3['_merge'] == 'left_only']
produit_jamais_vendu # c'est P5 et P6

nb_ligne_avant :  7
nb_ligne_après :  7
  order_id product_id  quantite  nom categorie  prix_num date_maj_ok  \
4       O4         P9         1  NaN       NaN       NaN         NaT   

      _merge  
4  left_only  

Agrégation par catégorie  : 
     categorie     ca  nb_ligne  nb_montant_connu
1       audio  179.7         3                 2
0  accessoire  113.7         3                 3


,product_id,nom,categorie,prix_num,date_maj_ok,order_id,quantite,_merge
3,P5,Webcam,video,NaN,2026-08-12,NaN,NaN,left_only
7,P6,Micro,audio,79.0,NaT,NaN,NaN,left_only


Partie D : Débogage

Ce script contient au moins 7 erreurs. Pour chaque ligne fautive : dis ce qui se passe (une erreur qui s'affiche, ou un résultat faux sans prévenir), explique pourquoi, puis corrige.

In [31]:
import pandas as pd
df = pd.read_csv("commandes.csv") # pas de précision sur le separateur ni sur le chargement des donnée uniquement en text risque de plantage
df.drop_duplicates(subset=["order_id"]) # suppresion de doublon perdu car pas d'affectation du résultat
df = df[df["statut"] == "completed" & df["canal"] == "web"]
df["montant"] = pd.to_numeric(df["montant"]) # s'il y a un montant qui n'est pas numérique le code plante, il faut ajouter errors='coerce'
df["date"] = pd.to_datetime(df["date_commande"], format="%Y-%m-%d", errors="coerce") # tous ce qui n'ont pas le format "%Y-%m-%d" 
# seront mis null même si la date existe

res = df.groupby("canal").agg(ca=("montant", "count")) # si c'est le ca qu'on veut calculer c'est pas count pas sum
res.to_csv("resultat.csv")

,order_id,product_id,quantite,nom,categorie,prix_num,date_maj_ok,_merge,montant
0,O1,P1,1,Casque,audio,59.9,2026-08-01,both,59.9
1,O1,P2,2,Souris,accessoire,22.9,2026-09-15,both,45.8
2,O2,P3,1,Clavier,accessoire,45.0,2026-08-15,both,45.0
3,O3,P1,2,Casque,audio,59.9,2026-08-01,both,119.8
4,O5,P2,1,Souris,accessoire,22.9,2026-09-15,both,22.9
5,O6,P4,3,Enceinte,audio,NaN,2026-08-10,both,NaN


In [45]:
# CORRECTION

import pandas as pd
df = pd.read_csv("commandes.csv", sep=';', dtype='str') # import pludent
avant = len(df)
df = df.drop_duplicates(subset=["order_id"]) # suppresion des doublons sur order_id
print("Doublons supprimés : ", avant - len(df)) 
df["montant"] = pd.to_numeric(df["montant"], errors='coerce')
d1 = pd.to_datetime(df["date_commande"], format="%Y-%m-%d", errors="coerce")
d2 = pd.to_datetime(df["date_commande"], format="%d/%m/%Y", errors="coerce")
df["date"] = d1.fillna(d2)          # prend d1, sinon d2
res = df.groupby("canal", dropna=False).agg(ca=("montant", "sum"))
print(res)
res.to_csv("resultat.csv", sep=';')

Doublons supprimés :  1
          ca
canal       
app    684.0
web    375.0
NaN      0.0
